# MiniChatGPT Lab

## Unidad 2 - Clase 2: La Revolucion de los Transformers

Construiras un **Mini ChatGPT** dentro de este notebook con GPT-2 (decoder-only):

- System prompt
- Historial de conversacion
- Generacion autoregresiva
- Temperatura, top-p y context window

> Ejecuta **Run All** desde arriba. Si cambias archivos en `src/`, reinicia kernel y vuelve a ejecutar.

## Objetivos
1. Cargar GPT-2 con Hugging Face Transformers.
2. Enviar mensajes y recibir respuestas.
3. Mantener historial multi-turno.
4. Experimentar con system prompt y muestreo.
5. Observar el limite de context window.

## Nota importante (GPT-2 vs ChatGPT)

GPT-2 **no** es ChatGPT: no tiene instrucciones ni RLHF. Fue entrenado para **continuar texto** en ingles.

- Usa preguntas en **ingles** en este notebook.
- El system prompt orienta el estilo, pero no garantiza obediencia perfecta.
- El objetivo es entender el **mecanismo**, no igualar ChatGPT comercial.

In [ ]:
%matplotlib inline

import importlib
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

warnings.filterwarnings('ignore', category=UserWarning)

def find_root(start: Path) -> Path:
    for folder in [start, *start.parents]:
        if (folder / 'src').exists() and 'MiniChatGPT' in folder.name:
            return folder
    raise FileNotFoundError('Abre el notebook desde MiniChatGPT_Lab/notebooks/')

ROOT = find_root(Path.cwd())
SRC = ROOT / 'src'
IMAGES = ROOT / 'images'
IMAGES.mkdir(parents=True, exist_ok=True)

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import chat_engine
import prompt_utils
import viz_utils

importlib.reload(prompt_utils)
importlib.reload(chat_engine)
importlib.reload(viz_utils)

from chat_engine import ChatConfig, MiniChatGPT
from viz_utils import plot_context_usage, plot_temperature_comparison

print('Entorno listo.')
print('ROOT =', ROOT)
print('Device se detectara al cargar el modelo.')

# Parte 1 — Cargar el modelo
GPT-2 es **decoder-only**: predice la siguiente palabra dado todo lo anterior.

In [ ]:
config = ChatConfig(
    model_name='gpt2',
    max_new_tokens=50,
    temperature=0.8,
    top_p=0.92,
    top_k=50,
    repetition_penalty=1.15,
    max_context_tokens=512,
)

chat = MiniChatGPT(
    config=config,
    system_prompt=(
        'You are a helpful AI tutor. '
        'Answer in 1-2 clear sentences about NLP and Transformers.'
    ),
)

print('Cargando GPT-2 (primera vez puede tardar)...')
chat.load()
print('MiniChatGPT listo en device:', chat._device)

# Parte 2 — Primer turno de chat
Un turno = User + Assistant.

In [ ]:
chat.reset()

pregunta_1 = 'What is a Transformer in machine learning?'
resultado_1 = chat.chat(pregunta_1)

print('Usuario:', resultado_1['user'])
print('Asistente:', resultado_1['assistant'])
print('Tokens entrada:', resultado_1['tokens_in'], '| salida:', resultado_1['tokens_out'])

In [ ]:
print('--- Prompt completo que vio el modelo ---')
print(resultado_1['prompt'])

# Parte 3 — Conversacion multi-turno
El historial se acumula como en ChatGPT.

In [ ]:
chat.reset()

turnos = [
    'Explain self-attention in simple words.',
    'Give me a one-sentence example.',
    'Why is it better than LSTM for long text?',
]

for pregunta in turnos:
    r = chat.chat(pregunta)
    print('User:', r['user'])
    print('Assistant:', r['assistant'])
    print('-' * 60)

display(chat.history_dataframe())

# Parte 4 — System prompt
Cambia la personalidad del asistente.

In [ ]:
chat.reset()
chat.system_prompt = (
    'You are a patient AI professor. Explain with simple analogies in 2 sentences.'
)

r_profesor = chat.chat('What is a token in NLP?')
print('System prompt = profesor:')
print(r_profesor['assistant'])

In [ ]:
chat.reset()
chat.system_prompt = 'You are a concise technical assistant. Maximum 2 short sentences.'

r_tecnico = chat.chat('What is a token in NLP?')
print('System prompt = tecnico:')
print(r_tecnico['assistant'])

# Parte 5 — Temperatura
Misma pregunta, distinta creatividad.

In [ ]:
pregunta_temp = 'Explain why Transformers replaced LSTM in NLP.'
temperaturas = [0.3, 0.8, 1.2]
comparacion = []

for temp in temperaturas:
    chat.reset()
    chat.config.temperature = temp
    r = chat.chat(pregunta_temp)
    comparacion.append({'temperature': temp, 'reply': r['assistant']})
    print(f"T={temp} -> {r['assistant']}\n")

plot_temperature_comparison(
    comparacion,
    save_path=str(IMAGES / 'temperatura_comparacion.png'),
)

# Parte 6 — Context window
Si el historial crece, el contexto se satura.

In [ ]:
chat.reset()
chat.system_prompt = 'You are a helpful assistant. Answer briefly.'

for i in range(6):
    chat.chat(f'Tell me fact number {i + 1} about neural networks.')

tokens_usados = chat.context_token_count('')
print('Tokens en historial:', tokens_usados)
print('Limite configurado:', chat.config.max_context_tokens)

plot_context_usage(
    tokens_usados,
    chat.config.max_context_tokens,
    save_path=str(IMAGES / 'context_window.png'),
)

# Parte 7 — Tu sesion libre
Edita `mis_preguntas` y ejecuta. Usa **ingles** para mejores resultados.

In [ ]:
chat.reset()
chat.system_prompt = (
    'You are MiniChatGPT, an educational NLP assistant. '
    'Answer clearly in 1-2 sentences.'
)
chat.config.temperature = 0.75
chat.config.top_p = 0.9

mis_preguntas = [
    'What is self-attention?',
    'Give an example with the ambiguous word bank.',
    'What is the difference between BERT and GPT?',
]

print('=' * 60)
print('MINI CHATGPT — SESION PERSONAL')
print('=' * 60)

for p in mis_preguntas:
    r = chat.chat(p)
    print(f"\nTu: {r['user']}")
    print(f"Bot: {r['assistant']}")
    print(f"(tokens in: {r['tokens_in']} | out: {r['tokens_out']})")

display(Markdown('### Historial completo'))
display(chat.history_dataframe())

## Cierre pedagogico

### Preguntas de reflexion
1. Que parte del flujo de ChatGPT replica tu Mini ChatGPT?
2. Para que sirve el system prompt?
3. Que cambio al subir la temperatura a 1.2?
4. Por que existe un limite de context window?
5. Que necesitarias para acercarte a ChatGPT real?

### Entregable
Compara tu Mini ChatGPT con ChatGPT comercial (200-300 palabras).

## Ejercicio extra (+10 puntos)
Prueba 3 system prompts (tutor, chef, programador) con la misma pregunta y guarda capturas en `images/`.